# Setup Environment and Clone Repository

### Guidelines for Generating a GitHub Personal Access Token (PAT)
Since your repository is private, you will need a token to clone it securely in this notebook:
1. Go to your Github **Settings** -> **Developer settings** -> **Personal access tokens** -> **Tokens (classic)**.
2. Click **Generate new token (classic)**.
3. Set a note (e.g., `Colab Training`), set an expiration, and check the `repo` scope to give it access to your private repositories.
4. Click **Generate token** and copy the token string. It will prompt you for this string in the cell below.


In [ ]:
import os
import sys
import getpass
import shutil
from pathlib import Path

# Set repository details
GITHUB_USERNAME = "arshadpatel2001"
REPO_NAME = "MLfTCC"

# Prompt securely for GitHub Personal Access Token
GITHUB_TOKEN = getpass.getpass("Enter your GitHub Personal Access Token: ")

# Construct the authenticated remote URL
repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(REPO_NAME):
    !git clone {repo_url}
else:
    print(f"Repository {REPO_NAME} already exists.")

# Navigate to the repository
# If the repository has a 'Project' sub-folder, cd into it
if os.path.exists(f"{REPO_NAME}/Project"):
    %cd {REPO_NAME}/Project
else:
    %cd {REPO_NAME}

# Clear the token from memory
del GITHUB_TOKEN


### Dataset Processing
To access the dataset from the shared drive folder `https://drive.google.com/drive/folders/1lYCyCtrKLFoDeRK_HyKCYOWiAzcmYEiz`:
1. Open the Drive folder link in your browser.
2. Click the folder name dropdown at the top and select **"Organize"** -> **"Add shortcut"** -> **"All locations"** -> **"My Drive"**.
3. Keep the shortcut name exactly the same (or adjust the `DRIVE_FOLDER` variable below to match what you named it).


In [ ]:
# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    print("Not running in Google Colab. Expecting local data usage.")
    IN_COLAB = False

# Update the path below to match where your Drive Shortcut is placed.
# e.g. /content/drive/MyDrive/1lYCyCtrKLFoDeRK_HyKCYOWiAzcmYEiz or whatever shortcut name you gave it.
DRIVE_FOLDER = "/content/drive/MyDrive/1lYCyCtrKLFoDeRK_HyKCYOWiAzcmYEiz"

DATA_ROOT = "./data/tropicyclonenet/TCND_test"
os.makedirs(DATA_ROOT, exist_ok=True)

if IN_COLAB:
    if os.path.exists(DRIVE_FOLDER):
        print(f"Looking for dataset zip files in: {DRIVE_FOLDER}")
        # Look for a zip file within the shared folder
        zip_file_paths = list(Path(DRIVE_FOLDER).glob("*.zip"))
        
        if zip_file_paths:
            dataset_zip = zip_file_paths[0]
            print(f"Extracting full dataset from {dataset_zip} to {DATA_ROOT}...")
            # Unzip dataset locally (-q for quiet, -n to not overwrite, -d to output dir)
            !unzip -n -q "{dataset_zip}" -d "{DATA_ROOT}"
            print("Extraction complete.")
        else:
            print("No .zip dataset file found in that Google Drive folder!")
    else:
        print(f"{DRIVE_FOLDER} does not exist. Please make sure the folder shortcut is placed in MyDrive.")


### Training Setup
Use the below block to kick off the training.


In [ ]:
# Run the desired training command
!python src/train.py \
    --mode lobo \
    --target_basin SI \
    --epochs 20 \
    --eval_every 5 \
    --data_root ./data/tropicyclonenet/TCND_test \
    --output_dir ./lobo_test_data


### Transfer Model Outputs to Google Drive
After training, copies model weights and logs generated in Colab back to your Google Drive folder. 


In [ ]:
if IN_COLAB and os.path.exists(DRIVE_FOLDER):
    print(f"Transferring logs and weights back to {DRIVE_FOLDER}...")
    
    # Copy the logs directory
    LOGS_DIR = "./logs"
    if os.path.exists(LOGS_DIR):
        dest_logs = os.path.join(DRIVE_FOLDER, "logs")
        shutil.copytree(LOGS_DIR, dest_logs, dirs_exist_ok=True)
        print("Successfully transferred logs folder.")
    else:
        print(f"{LOGS_DIR} not found.")

    # Copy the lobo_test_data outputs (weights)
    OUTPUT_DIR = "./lobo_test_data"
    if os.path.exists(OUTPUT_DIR):
        dest_outputs = os.path.join(DRIVE_FOLDER, "lobo_test_data")
        shutil.copytree(OUTPUT_DIR, dest_outputs, dirs_exist_ok=True)
        print("Successfully transferred output weights and results.")
    else:
        print(f"{OUTPUT_DIR} not found.")
        
    print("All requested files copied to Google Drive!")
else:
    print("Upload step skipped because it's not running in Colab or the Drive folder was not found.")
